# ⚡ EduVault Content Delivery System (CDS) — Cloud Supercharger
### High-Speed Automated Batch Ingestion & Cloud Transcoding Engine

> **Performance Notice:** Running on Google Colab provides a **1.5 to 2.5 Gbps network interface** with direct European peering to Telegram DCs (DC2/DC4). Upload speeds consistently achieve **45–80 MB/s**, completing a 970-lecture batch in **~5 hours** instead of 3 days on residential broadband.

---
### Architecture
- **Ingress CDN:** Native Linux `N_m3u8DL-RE` with binary fragment merging and automated retry.
- **Egress MTProto:** Pyrogram 4-Worker Turbo Upload with C-accelerated `tgcrypto` AES + warm session pooling.
- **State Persistence:** Supabase Cloud PostgreSQL (`video_assets` table) — progress survives runtime resets.

## Step 1 · Clone Repository & Provision Cloud Environment
Clones `ArYaN-X404/CDS` into `/content/CDS`, sets it as the working directory for all cells, installs `ffmpeg`, `aria2`, `N_m3u8DL-RE`, and `tgcrypto`.

In [ ]:
import os, sys, subprocess

REPO_DIR = '/content/CDS'
REPO_URL = 'https://github.com/ArYaN-X404/CDS.git'

# --- Clone if not already present ---
if not os.path.isdir(os.path.join(REPO_DIR, 'scripts')):
    print('📥 Cloning EduVault CDS repository...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('❌ Clone failed:', result.stderr)
        raise RuntimeError('Repository clone failed')
    print('✅ Repository cloned to', REPO_DIR)
else:
    print('✅ Repository already present at', REPO_DIR)
    # Pull latest changes
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], capture_output=True)

# --- Set working directory for ALL cells in this session ---
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'📂 Working directory: {os.getcwd()}')
print(f'📁 Contents: {os.listdir(".")}')

# --- Run the automated cloud provisioner ---
print('\n⚙️ Running cloud environment provisioner...')
!python scripts/setup_cloud.py

## Step 2 · Credential Injection
Loads API keys from **Google Colab Secrets** (recommended) or prompts interactively.

> **💡 Tip:** Add your secrets once via **🔑 Secrets** (left sidebar) — they persist across sessions and are never stored in the notebook.

In [ ]:
import os, getpass

# Load from Colab Secrets (preferred) or fall back to env / interactive prompt
try:
    from google.colab import userdata
    get_secret = userdata.get
except Exception:
    get_secret = lambda k: None

def resolve_credential(key, prompt, default=''):
    for source in [lambda: get_secret(key), lambda: os.environ.get(key)]:
        try:
            v = source()
            if v: return v
        except Exception:
            pass
    if not default:
        return getpass.getpass(f'Enter {key} ({prompt}): ')
    return default

os.environ['API_ID']             = str(resolve_credential('API_ID',            'Telegram API ID'))
os.environ['API_HASH']           = str(resolve_credential('API_HASH',           'Telegram API Hash'))
os.environ['BOT_TOKEN']          = str(resolve_credential('BOT_TOKEN',          'Telegram Bot Token'))
os.environ['CHANNEL_ID']         = str(resolve_credential('CHANNEL_ID',         'Storage Channel ID', '-1004460495677'))
os.environ['OWNER_ID']           = str(resolve_credential('OWNER_ID',           'Owner Telegram User ID', '7507697121'))
os.environ['FAST_UPLOAD_WORKERS']= str(resolve_credential('FAST_UPLOAD_WORKERS','Pyrogram Upload Workers', '14'))
os.environ['SUPABASE_URL']       = str(resolve_credential('SUPABASE_URL',       'Supabase URL', 'https://fytemrmebqovsclumwqu.supabase.co'))
os.environ['SUPABASE_KEY']       = str(resolve_credential('SUPABASE_KEY',       'Supabase Service Role Key'))

print('\n✅ All credentials loaded!')
print(f'   API_ID      : {os.environ["API_ID"][:4]}****')
print(f'   CHANNEL_ID  : {os.environ["CHANNEL_ID"]}')
print(f'   SUPABASE_URL: {os.environ["SUPABASE_URL"]}')

## Step 3 · Background Keepalive & Disconnect Guard
Prevents Google Colab from disconnecting after 90 minutes of inactivity.

In [ ]:
import time, threading, sys

def _keepalive():
    while True:
        time.sleep(55)
        sys.stdout.write('\r[HEARTBEAT] Pipeline active. Uplink open.\n')
        sys.stdout.flush()

threading.Thread(target=_keepalive, daemon=True).start()
print('🛡️  Colab Keepalive Daemon active.')

## Step 4 · Diagnostic Verification
Confirms Supabase Cloud REST connectivity, SQLite state, and current upload progress.

In [ ]:
import os
# Ensure we're still in the repo root (safe to re-run)
if not os.path.exists('scripts/fast_codebase_tool.py'):
    os.chdir('/content/CDS')
!python scripts/fast_codebase_tool.py diag

## Step 5 · Launch Ingestion Pipeline
Starts the Cloud Worker Agent. Streams **live telemetry** back to your local monitoring terminal.

In [ ]:
import os
if not os.path.exists('scripts/cloud_worker.py'):
    os.chdir('/content/CDS')

# ▶ Run Remote Cloud Worker Agent
#   Streams live telemetry → local orchestrator terminal
!python scripts/cloud_worker.py